# Token缓存

大部分AI创业公司都不是死于模型太差，而是死于账单太贵。

## 基本问题

用户问了同一个问题，只是措辞少有不同。你的提供提示词每次请求都在注入。上下文RAG检索每次也在注入。

你在为重复的计算支付全额开销。

## 基本概念

### LLM调用的开销构成

从用户输入到组装系统提示词、RAG检索、再是用户提问构建大模型输入，这部分按照输入标准词元收费。模型处理后生成的结果按照输出标准词元收费。

系统提示词是无形的账单杀手，如果不加处理每次请求都需要为这一前缀支付账单。

### 供应商缓存

三个主流的模型供应商都支持供应商缓存，但是机制有所不同。
|供应商|机制|折扣|最小限制|缓存时限|
|---|---|---|---|---|
|Anthropic|显事的cache_control标记|写的时候多花25%，命中时省90%|1024/2048词元|默认5分钟，超过一小时溢价|
|OpenAI|自动前缀匹配|命中时省50%|1024词元|最大限度到一小时|
|Gemini|显式CachedContext API|省75%|4096/32768词元|用户可以配置过期时间|

### 语义缓存————你自定义的层

供应商缓存只在相同前缀条件下工作，语义缓存处理更复杂的场景：语句不通，意思相同。

将查询做语义嵌入，对于新的查询，如果相似度大于阈值（通常0.92～0.95），就返回缓存的回复。

嵌入开销基本可以忽略不计。

### 精确缓存————哈希比较

对于确定性的调用（温度为0，相同模型，相同提示词），精确缓存更简单也更快。将完整提示词作哈希，然后检查缓存，如果找到了就直接返回。

适用于：
1. 系统提示词+固定上下文+相同的用户查询
2. Funtion calling 相同的工具定义
3. 同一个文档被处理多次的批处理过程

### 限流————保护你的钱包

限流不只关乎公平，它关乎到你的钱包。

### 模型路由————正确的模型做正确的事

不是所有的查询都需要前沿顶尖模型。使用一个简单的分类器将简单的查询分配给便宜的模型，复杂的则留给贵的模型。

### 开销跟踪————知道你的钱花去了哪

对于API调用，记录:时间戳、模型名、输入词元数、输出词元数、延迟、费用、用户名、缓存命中情况、请求类型。

### 优化栈

按顺序应用这些技术，每层都聚合在上一层：
1. 供应商缓存。  加缓存标记，省30%-50%。
2. 精确缓存。    哈希和字典，省10%-20%。
3. 语义缓存。    嵌入和相似度比较，省15%-30%。
4. 模型路由。    靠分类器，省40%-70%
5. 限流。        保护钱包
6. 压缩提示词。  重写提示词，省10%-30%
7. 批请求。      靠batch API，闲时省50%



# API

In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage

import sys
from pathlib import Path
from rich import print as rprint

sys.path.append(str(Path("../../00_Common").resolve()))

from user_tools import SectionPrinter, load_project_env

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"

# 前缀够长才更容易触发供应商前缀缓存（DeepSeek 自动前缀匹配）
SYSTEM_PROMPT = (
    "You are a helpful customer support agent for ShopLite. "
    "Always answer politely, briefly, and in English. "
    "Return policy: customers may return unused items within 30 days of delivery "
    "with the original receipt; refunds go to the original payment method within "
    "5–10 business days; opened software, gift cards, and final-sale items are "
    "non-returnable; shipping for returns is paid by the customer unless the item "
    "was defective or incorrect; exchanges are allowed for size/color when stock "
    "is available. "
    "Warranty: 1-year limited warranty on electronics against manufacturing defects. "
    "Shipping: standard 3–5 business days, express 1–2 business days in-region. "
    "Tone rules: do not invent order IDs; if data is missing, ask one clarifying "
    "question; never promise discounts not listed in policy. "
) * 8  # 拉长前缀，便于观察 cached tokens

agent = create_agent(
    model=init_chat_model(MODEL, extra_body={"thinking": {"type": "disabled"}}),
    system_prompt=SYSTEM_PROMPT,
)


def print_token_usage(label: str, response: dict) -> None:
    ai = next(
        m for m in reversed(response["messages"]) if isinstance(m, AIMessage)
    )
    usage = ai.response_metadata.get("token_usage") or {}
    details = usage.get("prompt_tokens_details") or {}
    prompt_tokens = usage.get("prompt_tokens")
    cached_tokens = details.get("cached_tokens", 0)
    completion_tokens = usage.get("completion_tokens")
    with SectionPrinter(label):
        rprint(f"prompt_tokens     : {prompt_tokens}")
        rprint(f"cached_tokens     : {cached_tokens}")
        rprint(f"completion_tokens : {completion_tokens}")
        rprint(ai.content)


messages = [{"role": "user", "content": "What is the return policy?"}]

# 第 1 次：写入前缀缓存；第 2 次：同前缀应出现 cached_tokens
response1 = agent.invoke({"messages": messages})
print_token_usage("call #1 (cold / write cache)", response1)

response2 = agent.invoke({"messages": messages})
print_token_usage("call #2 (warm / read cache)", response2)

InvalidUpdateError: Expected dict, got [{'role': 'system', 'content': 'You are a helpful customer support agent...'}, {'role': 'user', 'content': 'What is the return policy?'}]
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE